# Train the U-Net on S1S2-Water chips (Kaggle GPU)

Rebuilds the chips from the pinned Hugging Face mirror, checks **every chip against the committed
`data/manifest.csv` by content hash**, trains the U-Net once with the configuration frozen in
`PREREG.md`, picks the checkpoint on the validation split, and scores `val` and `test` once.
Nothing from the dataset is stored in the repo; the manifest decides what is a valid chip.

**Notebook settings before *Run all*:**

1. *Accelerator*: **GPU T4**.
2. *Internet*: **on** (pip, GitHub, Hugging Face, pretrained ResNet-34 weights).
3. *Add-ons -> Secrets*: none needed.
4. Fill in `REPO_REF` in the first code cell with the commit to run; the install cell refuses
   to run while it is still the placeholder.

**Test discipline.** The best checkpoint is chosen on `val` only. `test` is scored once, at the
end, by the checkpoint `val` chose, and the scoring cell refuses to overwrite an existing test
result. Nothing in this notebook is changed because of a test number; a rerun with different
settings is an amendment to `PREREG.md`, reported next to the original result.

**Output** (`/kaggle/working/`): `results/` (per-chip CSVs and the run record, small, committed to
the repo), `masks/` (predicted masks for overlays), `run/best.pt` (the checkpoint, uploaded to the
Hugging Face Hub, not committed).

In [ ]:
# --- Run configuration: everything that defines this run is here -------------------------
REPO_URL = "https://github.com/Prithv122/segmentation.git"
REPO_REF = (
    "<fill in: commit sha>"  # the commit under test; the install cell rejects this placeholder
)

# Frozen in PREREG.md ("U-Net training"); must equal the TrainConfig defaults.
TRAIN = {
    "encoder": "resnet34",
    "encoder_weights": "imagenet",
    "epochs": 40,
    "batch_size": 16,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "crop": 256,
    "patience": 8,
    "seed": 0,
}
NUM_WORKERS = 4  # DataLoader workers for training
CHIP_WORKERS = 8  # parallel scene downloads while rebuilding chips

## 1. GPU check

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 and rerun"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Install and clone

In [ ]:
%pip install -q segmentation-models-pytorch==0.5.0 rasterio==1.5.2 remotezip==0.12.6

In [ ]:
import os
import sys
from pathlib import Path

assert not REPO_REF.startswith("<"), "set REPO_REF to a commit sha first"

REPO = Path("/tmp/segmentation")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", REPO_REF], check=True)
COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True
)
sys.path.insert(0, str(REPO / "src"))  # a running kernel does not see the new editable install
os.environ["SEGMENT_GIT_COMMIT"] = COMMIT
print("repo at", COMMIT)

## 3. Rebuild the chips and verify them against the committed manifest

Chips go to `/kaggle/temp/data` (falling back to `/tmp/data`): it is writable, outside the
20 GB `/kaggle/working` quota, and not saved as notebook output, so the chip files are not
re-hosted. The build takes about 15 minutes. Any chip that is missing or whose content hash
differs from `data/manifest.csv` stops the notebook.

In [ ]:
import shutil

DATA = Path("/kaggle/temp/data") if Path("/kaggle/temp").is_dir() else Path("/tmp/data")
DATA.mkdir(parents=True, exist_ok=True)
shutil.copy(REPO / "data" / "scenes.csv", DATA / "scenes.csv")


def run_segment(*args: str) -> None:
    subprocess.run([sys.executable, "-m", "segment.cli", *args], check=True, cwd=REPO)


run_segment(
    "-v",
    "chips",
    "--scenes",
    str(DATA / "scenes.csv"),
    "--root",
    str(DATA),
    "--workers",
    str(CHIP_WORKERS),
)

In [ ]:
from segment.chips import verify_chips
from segment.data import check_splits, read_manifest

MANIFEST = REPO / "data" / "manifest.csv"
committed = read_manifest(MANIFEST)
rebuilt = read_manifest(DATA / "manifest.csv")

assert not check_splits(committed), check_splits(committed)
extra = {r.chip_id for r in rebuilt} - {r.chip_id for r in committed}
problems = verify_chips(committed, DATA)
assert not problems and not extra, (problems[:10], len(problems), sorted(extra)[:10])
print(f"{len(committed)} chips match data/manifest.csv by content hash; splits clean")

## 4. Train

One run with the frozen configuration. The test split is never opened here (asserted in code).

In [ ]:
import json

from segment.train import TrainConfig, train

RUN_DIR = Path("/kaggle/working/run")
cfg = TrainConfig(
    chips_dir=DATA,
    manifest=MANIFEST,
    out_dir=RUN_DIR,
    num_workers=NUM_WORKERS,
    device="cuda",
    **TRAIN,
)
record = train(cfg)

print(f"{'epoch':>5} {'train_loss':>10} {'val_iou':>8} {'lr':>9} {'sec':>6}")
for e in record["epochs"]:
    print(
        f"{e['epoch']:>5} {e['train_loss']:>10.4f} {e['val_iou']:>8.4f} "
        f"{e['lr']:>9.2e} {e['seconds']:>6.1f}"
    )
print(f"best epoch {record['best_epoch']}, val IoU {record['best_val_iou']:.4f}")

## 5. Score val and test, once

Uses `run/best.pt`, the checkpoint `val` chose. The cell refuses to run if a test result already
exists in this session.

In [ ]:
RESULTS = Path("/kaggle/working/results")
MASKS = Path("/kaggle/working/masks")
assert not (RESULTS / "per_chip_unet_test.csv").exists(), "test was already scored"

run_segment(
    "predict",
    "--checkpoint",
    str(RUN_DIR / "best.pt"),
    "--manifest",
    str(MANIFEST),
    "--root",
    str(DATA),
    "--splits",
    "val",
    "test",
    "--results",
    str(RESULTS),
    "--masks-dir",
    str(MASKS),
    "--device",
    "cuda",
)

## 6. Run record and outputs

In [ ]:
final = json.loads((RUN_DIR / "run_record.json").read_text(encoding="utf-8"))
final["notebook"] = {
    "repo_url": REPO_URL,
    "repo_commit": COMMIT,
    "chips_verified": len(committed),
    "chip_root": DATA.as_posix(),
}
(RESULTS / "run_record.json").write_text(json.dumps(final, indent=2) + "\n", encoding="utf-8")

for path in sorted(p for d in (RESULTS, RUN_DIR, MASKS) for p in d.rglob("*") if p.is_file()):
    print(f"{path.stat().st_size / 1e6:>9.2f} MB  {path}")

## Download and commit

From this notebook's output, download:

- `results/per_chip_unet_val.csv` and `results/per_chip_unet_test.csv`
- `results/run_record.json`
- `masks/masks_unet_val.npz` and `masks/masks_unet_test.npz` (for overlays)
- `run/best.pt`

Commit the CSVs and `run_record.json` under `results/` in the repo, then run `segment eval` to build
`results/summary.md`. The masks are only needed locally for overlays. `best.pt` goes to the
Hugging Face Hub, not to git.